# Caso práctico: ¿Quién va a cancelar Netflix?

Flujo reproducible de revisión, limpieza, análisis exploratorio, modelado, evaluación y generación de entregables. Solo se usa el CSV proporcionado (`neflix.csv`). Los resultados describen este dataset académico y no demuestran causalidad ni validez en producción.

## 1. Revisión inicial

Se examinan dimensiones, tipos, nulos, duplicados exactos, ids repetidos, categorías, rangos numéricos y distribución del objetivo. La comprobación de forma contrasta el archivo real con el entregado para la actividad.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'raw' / 'neflix.csv').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'raw' / 'neflix.csv').exists():
    raise FileNotFoundError('No se encontró data/raw/neflix.csv. Abra el notebook desde la carpeta del proyecto.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Proyecto: {ROOT}')

In [ ]:
raw_path = ROOT / 'data' / 'raw' / 'neflix.csv'
df_original = pd.read_csv(raw_path)
assert df_original.shape == (5025, 20), f'Forma inesperada del CSV: {df_original.shape}'
print('Dimensiones:', df_original.shape)
display(df_original.dtypes.rename('tipo').to_frame())
print('\nNulos por variable:')
display(df_original.isna().sum().rename('nulos').to_frame())
print(f'Duplicados exactos: {df_original.duplicated().sum():,}')
print(f'Repeticiones de id_usuario después de la primera: {df_original.id_usuario.duplicated().sum():,}')
print('\nCategorías de texto:')
for column in df_original.select_dtypes(include='object').columns:
    print(f'\n{column} ({df_original[column].nunique(dropna=True)} categorías)')
    print(df_original[column].value_counts(dropna=False).to_string())
numeric_columns = df_original.select_dtypes(include='number').columns
print('\nRangos numéricos:')
display(df_original[numeric_columns].describe().T[['min', '50%', 'max']].rename(columns={'50%': 'mediana'}))
print('\nObjetivo original:')
display(df_original.cancelo.value_counts(dropna=False).rename_axis('cancelo').to_frame('usuarios'))
fechas = pd.to_datetime(df_original.fecha_registro, errors='coerce')
print(f'Fechas válidas: {fechas.notna().sum():,}; inválidas: {fechas.isna().sum():,}; rango: {fechas.min()} a {fechas.max()}')

## 2. Limpieza y auditoría

Se eliminan copias exactas; si hubiera registros diferentes del mismo usuario, se retiene el más completo y luego el más reciente. Las reglas de dominio convierten valores imposibles a nulos, sin eliminar filas por tener datos faltantes. La auditoría registra cantidades, tratamientos y justificaciones. Las imputaciones de predictores se aprenden después dentro de cada pipeline.

In [ ]:
from src.limpieza import cargar_y_limpiar, exportar_datos_limpios

df_limpio, auditoria = cargar_y_limpiar(raw_path)
clean_path = ROOT / 'data' / 'processed' / 'netflix_limpio.csv'
exportar_datos_limpios(df_limpio, auditoria, clean_path)
print(f'Filas limpias: {len(df_limpio):,}; ids únicos: {df_limpio.id_usuario.nunique():,}')
print('Faltantes conservados para tratamiento posterior:')
display(df_limpio.isna().sum().loc[lambda values: values.gt(0)].rename('nulos').to_frame())
print('Auditoría de limpieza:')
display(auditoria)
assert df_limpio.id_usuario.is_unique
assert set(df_limpio.cancelo.dropna().unique()) <= {0, 1}

## 3. Análisis exploratorio

Las tasas muestran porcentaje y tamaño de grupo. Los resúmenes de comportamiento informan cuántas observaciones válidas se usan; se presentan medianas por la asimetría potencial de las distribuciones.

In [ ]:
from src.visualizacion import generar_graficos_exploratorios, interpretacion_tasa, tasa_cancelacion

df_eda = df_limpio.loc[df_limpio.cancelo.notna()].copy()
cancelados = int(df_eda.cancelo.sum())
total_usuarios = len(df_eda)
print(f'Tasa general: {cancelados:,}/{total_usuarios:,} = {cancelados / total_usuarios:.2%}')
group_tables = {}
tables_dir = ROOT / 'reports' / 'tablas'
tables_dir.mkdir(parents=True, exist_ok=True)
for variable in ['plan', 'pais', 'metodo_pago', 'dispositivo_principal']:
    table = tasa_cancelacion(df_eda, variable).sort_values('tasa_cancelacion', ascending=False)
    group_tables[variable] = table
    table.to_csv(tables_dir / f'tasa_cancelacion_{variable}.csv', index=False)
    print(f'\nCancelación por {variable}:')
    display(table.assign(porcentaje=table.tasa_cancelacion.mul(100).round(2))[[variable, 'usuarios', 'cancelados', 'porcentaje']])
    print(interpretacion_tasa(table, variable))

behavior = ['horas_vistas_mes', 'dias_desde_ultimo_acceso', 'calificacion_promedio', 'tickets_soporte']
print('\nMedianas y datos válidos por estado de cancelación:')
behavior_summary = df_eda.groupby('cancelo')[behavior].agg(['count', 'median']).rename(index={0: 'No cancela', 1: 'Sí cancela'})
behavior_summary.to_csv(tables_dir / 'resumen_comportamiento_cancelacion.csv')
display(behavior_summary)
figures_dir = ROOT / 'reports' / 'figures'
figure_paths = generar_graficos_exploratorios(df_eda, figures_dir)
for figure_path in figure_paths:
    display(Image(filename=str(figure_path)))

## 4. Modelado y evaluación

Se reserva 25 % para prueba y se estratifica por `cancelo`. La validación cruzada de cinco folds para comparar F1 ocurre dentro del 75 % de entrenamiento; preprocesadores se ajustan en cada fold y el conjunto de prueba no se usa para ajustar umbrales ni variables. Falsos positivos son usuarios marcados para retención que no cancelan; falsos negativos son cancelaciones no detectadas.

In [ ]:
from src.modelado import entrenar_modelos, preparar_dataset

X_train, y_train, X_test, y_test = preparar_dataset(df_limpio)
assert len(X_train) == int(len(df_eda) * 0.75)
assert len(X_test) == len(df_eda) - len(X_train)
assert 'id_usuario' not in X_train and 'fecha_registro' not in X_train
print(f'Entrenamiento: {len(X_train):,}; prueba: {len(X_test):,}')
print('Clase positiva=1; distribución real:')
display(df_eda.cancelo.value_counts().rename(index={0: 'No cancela (0)', 1: 'Sí cancela (1)'}).to_frame('usuarios'))
print('Una exactitud alta puede ser engañosa si se aproxima a la proporción de la clase mayoritaria y el recall de cancelación es bajo.')
tabla_modelos, importancias = entrenar_modelos(df_limpio)
print('\nComparación (métricas de prueba y F1 de CV solo en entrenamiento):')
display(tabla_modelos)
for model_name in ['dummyclassifier', 'logisticregression', 'randomforestclassifier']:
    report_path = ROOT / 'reports' / 'metrics' / f'{model_name}_report.json'
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(f'\n{model_name}: report por clase y matriz [No, Sí]')
    display(pd.DataFrame(report['classification_report']).T)
    print(report['confusion_matrix'])
    tn, fp = report['confusion_matrix'][0]
    fn, tp = report['confusion_matrix'][1]
    print(f'Falsos positivos: {fp} (acción de retención innecesaria); falsos negativos: {fn} (cancelaciones no detectadas).')
    display(Image(filename=str(figures_dir / f'matriz_confusion_{model_name}.png')))
print('\nImportancia por permutación (F1, entrenamiento):')
display(importancias)
display(Image(filename=str(figures_dir / 'variables_importantes.png')))
evaluation = json.loads((ROOT / 'reports' / 'metrics' / 'evaluation_netflix.json').read_text(encoding='utf-8'))
print(evaluation['temporal_limitation'])
print('La importancia predictiva no demuestra causalidad.')

## 5. Hallazgos y recomendaciones

Los hallazgos se calculan desde el CSV limpio y las métricas ejecutadas en esta sesión. Las acciones propuestas son hipótesis para probar con grupo de control; cada una incluye un indicador y no presupone eficacia.

In [ ]:
print(f'1. Canceló {cancelados:,} de {total_usuarios:,} usuarios ({cancelados / total_usuarios:.1%}) en el dataset.')
for number, variable in enumerate(['plan', 'pais', 'metodo_pago', 'dispositivo_principal'], start=2):
    print(f'{number}. {interpretacion_tasa(group_tables[variable], variable)}')
print('\nAcciones hipotéticas e indicadores:')
median_by_class = df_eda.groupby('cancelo')[behavior].median()
if median_by_class.loc[1, 'dias_desde_ultimo_acceso'] > median_by_class.loc[0, 'dias_desde_ultimo_acceso']:
    print('- Probar reactivación por inactividad; indicador: cancelación y retención a 30 días frente a un grupo de control.')
if median_by_class.loc[1, 'horas_vistas_mes'] < median_by_class.loc[0, 'horas_vistas_mes']:
    print('- Probar recomendaciones/recordatorios para usuarios con menor actividad; indicador: horas vistas y cancelación a 30 días.')
if median_by_class.loc[1, 'tickets_soporte'] > median_by_class.loc[0, 'tickets_soporte']:
    print('- Revisar resolución de incidencias en soporte; indicador: resolución en primer contacto y cancelación posterior.')
print('- Para cada segmento con tasa descriptiva elevada, validar primero tamaño, periodo y estabilidad; indicador: diferencia de cancelación vs control.')
print('\nGeneración de presentación:')
from src.presentacion import build_presentation
presentation_path = build_presentation()
print(presentation_path)